In [ ]:
# Cell 1: (no-op — repo is public, no PAT needed)
print('Repo is public — skipping secrets setup')


In [ ]:
# Cell 2: Clone repo
%cd /kaggle/working
!rm -rf code-7f2
!git clone https://github.com/almaas-izdihar/7f2687fb74 code-7f2
%cd code-7f2
!git checkout experiment/vanilla-emaskd
!git log --oneline -5

In [ ]:
# Cell 3: Verify GPU
!nvidia-smi

In [ ]:
# Cell 3.5: Run Config
END_EPOCH       = 2     # 2 = smoke test, 200 = full run
BATCH_PER_GPU   = 128   # each GPU runs independently
WORKERS_PER_GPU = 4
SEED_GPU0       = 2024  # seed for GPU 0
SEED_GPU1       = 2025  # seed for GPU 1
EHSKD           = True  # True = EMA-SKD, False = baseline CE only

_mode = 'EMA-SKD' if EHSKD else 'Baseline'
print(f'Mode={_mode}  seeds=({SEED_GPU0}, {SEED_GPU1})  batch_per_gpu={BATCH_PER_GPU}  epochs={END_EPOCH}')

In [ ]:
# Cell 3.6: Pre-download CIFAR-100 + start GPU monitor
import torchvision, os, subprocess, time

os.makedirs('/kaggle/working/data', exist_ok=True)
# Toronto mirror unreliable — attach cifar-100-python Kaggle dataset instead
# torchvision.datasets.CIFAR100(root='/kaggle/working/data', train=True,  download=True)
# torchvision.datasets.CIFAR100(root='/kaggle/working/data', train=False, download=True)
print('CIFAR-100 ready (loaded from attached dataset).')

# Start background GPU sampler — runs across Cell 4 and Cell 5
_gpu_log = '/kaggle/working/gpu_log.csv'
_gpu_proc = subprocess.Popen(
    f'nvidia-smi --query-gpu=timestamp,index,utilization.gpu,memory.used '
    f'--format=csv,noheader,nounits -l 5 > {_gpu_log}',
    shell=True
)
print(f'GPU logger started (PID {_gpu_proc.pid}) → {_gpu_log}')

In [ ]:
# Cell 4: Run same experiment × 2 seeds in parallel (GPU 0 = SEED_GPU0, GPU 1 = SEED_GPU1)
import subprocess, os, glob, time, sys

bs   = str(BATCH_PER_GPU)
ep   = str(END_EPOCH)
wk   = str(WORKERS_PER_GPU)
mode = '--EHSKD --beta 0.5' if EHSKD else ''
tag  = 'emaskd' if EHSKD else 'baseline'

def build_cmd(gpu, seed):
    return (
        f'CUDA_VISIBLE_DEVICES={gpu} python main.py '
        f'--data_type cifar100 --data_path /kaggle/input/cifar-100-python '
        f'--classifier_type ResNet18 --batch_size {bs} '
        f'--end_epoch {ep} --workers {wk} --seed {seed} '
        f'{mode} '
        f'--experiment_type {tag}_seed{seed} '
        f'> /kaggle/working/stdout_gpu{gpu}.txt 2>&1'
    )

p0 = subprocess.Popen(build_cmd(0, SEED_GPU0), shell=True)
p1 = subprocess.Popen(build_cmd(1, SEED_GPU1), shell=True)
print(f'GPU 0: {tag} seed={SEED_GPU0}  PID={p0.pid}')
print(f'GPU 1: {tag} seed={SEED_GPU1}  PID={p1.pid}')
sys.stdout.flush()

def last_val(seed):
    logs = sorted(glob.glob(f'models/*seed{seed}*/log/log.txt'))
    if not logs:
        return 'no log yet'
    lines = [l for l in open(logs[-1]) if '[val]' in l]
    return lines[-1].strip() if lines else 'no val yet'

POLL = 60
while p0.poll() is None or p1.poll() is None:
    time.sleep(POLL)
    print(f'[{time.strftime("%H:%M:%S")}] seed{SEED_GPU0} {"✓" if p0.poll() is not None else "…"} {last_val(SEED_GPU0)}')
    print(f'[{time.strftime("%H:%M:%S")}] seed{SEED_GPU1} {"✓" if p1.poll() is not None else "…"} {last_val(SEED_GPU1)}')
    print()
    sys.stdout.flush()

p0.wait(); p1.wait()
print('Both seeds complete.')
sys.stdout.flush()

In [ ]:
# Cell 5: (no-op)
print('Both seeds launched in Cell 4.')

In [ ]:
# Cell 6: Metrics — parse both seed logs and print side-by-side
import glob, re, pandas as pd

def parse_log(path):
    rows = []
    with open(path) as f:
        for line in f:
            if '[val]' not in line:
                continue
            def g(key):
                m = re.search(rf'\[{key} ([^\]]+)\]', line)
                return float(m.group(1)) if m else None
            ep = re.search(r'\[Epoch (\d+)\]', line)
            if not ep:
                continue
            rows.append({
                'epoch':    int(ep.group(1)),
                'top1':     g('val_top1_acc'),
                'top5':     g('val_top5_acc'),
                'val_loss': g('val_loss'),
                'ece':      g('ECE'),
                'aurc':     g('AURC'),
                'eaurc':    g('EAURC'),
            })
    return pd.DataFrame(rows).set_index('epoch')

def find_log(seed):
    matches = sorted(glob.glob(f'models/*seed{seed}*/log/log.txt'))
    if not matches:
        raise FileNotFoundError(f'No log for seed {seed}')
    return matches[-1]

log0 = find_log(SEED_GPU0)
log1 = find_log(SEED_GPU1)
df0  = parse_log(log0)
df1  = parse_log(log1)

print(f'seed {SEED_GPU0}: {log0}  ({len(df0)} epochs)')
print(f'seed {SEED_GPU1}: {log1}  ({len(df1)} epochs)')
print()

metrics = ['top1', 'top5', 'ece', 'aurc', 'eaurc']
labels  = ['Top-1 (%)', 'Top-5 (%)', 'ECE (↓)', 'AURC (↓)', 'EAURC (↓)']

rows = []
for m, lbl in zip(metrics, labels):
    v0 = df0[m].iloc[-1] if m in df0 else float('nan')
    v1 = df1[m].iloc[-1] if m in df1 else float('nan')
    rows.append({'Metric': lbl, f'seed {SEED_GPU0}': v0, f'seed {SEED_GPU1}': v1})

summary = pd.DataFrame(rows)
print(summary.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print()
print(f'Paper target (EMA-SKD, 200ep): 79.19 ± 0.15')

In [ ]:
# Cell 7: Training curves — both seeds overlaid
import glob, re, matplotlib.pyplot as plt, matplotlib.ticker as ticker, pandas as pd

def _parse_log(path):
    rows = []
    with open(path) as f:
        for line in f:
            if '[val]' not in line: continue
            def g(key):
                m = re.search(rf'\[{key} ([^\]]+)\]', line)
                return float(m.group(1)) if m else None
            ep = re.search(r'\[Epoch (\d+)\]', line)
            if not ep: continue
            rows.append({'epoch': int(ep.group(1)), 'top1': g('val_top1_acc'),
                         'val_loss': g('val_loss'), 'ece': g('ECE'), 'aurc': g('AURC')})
    return pd.DataFrame(rows).set_index('epoch')

def find_log(seed):
    matches = sorted(glob.glob(f'models/*seed{seed}*/log/log.txt'))
    if not matches: raise FileNotFoundError(f'No log for seed {seed}')
    return matches[-1]

_df0 = _parse_log(find_log(SEED_GPU0))
_df1 = _parse_log(find_log(SEED_GPU1))

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
_mode = 'EMA-SKD' if EHSKD else 'Baseline'
fig.suptitle(f'{_mode} — CIFAR-100 / ResNet18 — 2 seeds', fontsize=13)

panels = [('top1', 'Top-1 Accuracy (%)'), ('val_loss', 'Val Loss'),
          ('ece',  'ECE (↓)'),            ('aurc',     'AURC (↓)')]

for ax, (col, title) in zip(axes.flat, panels):
    ax.plot(_df0.index, _df0[col], label=f'seed {SEED_GPU0}', linewidth=1.5)
    ax.plot(_df1.index, _df1[col], label=f'seed {SEED_GPU1}', linewidth=1.5, linestyle='--')
    ax.set_title(title); ax.set_xlabel('Epoch')
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/eval_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: /kaggle/working/eval_curves.png')

In [ ]:
# Cell 8: Resource usage — GPU utilization & memory timeseries
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Stop GPU logger
try:
    _gpu_proc.terminate()
    print(f'GPU logger stopped (PID {_gpu_proc.pid})')
except Exception as e:
    print(f'GPU logger stop: {e}')

# Parse CSV
df_gpu = pd.read_csv(
    '/kaggle/working/gpu_log.csv',
    names=['timestamp', 'gpu', 'util_pct', 'mem_mib'],
    skipinitialspace=True
)
df_gpu['timestamp'] = pd.to_datetime(df_gpu['timestamp'], format='%Y/%m/%d %H:%M:%S.%f', errors='coerce')
df_gpu = df_gpu.dropna(subset=['timestamp'])
df_gpu['t'] = (df_gpu['timestamp'] - df_gpu['timestamp'].min()).dt.total_seconds()
df_gpu['gpu'] = df_gpu['gpu'].astype(int)

# Save full numeric log as tsv (timestamp + t_sec + gpu + util + mem)
df_gpu.to_csv('/kaggle/working/gpu_log_parsed.tsv', sep='\t', index=False, float_format='%.1f')

# Save summary stats
stats = df_gpu.groupby('gpu')[['util_pct', 'mem_mib']].describe().round(1)
summary_lines = [
    f'GPU logger samples : {len(df_gpu)}',
    f'Total duration     : {df_gpu["t"].max():.0f}s',
    '',
    str(stats),
]
summary_text = '\n'.join(summary_lines)
with open('/kaggle/working/gpu_stats.txt', 'w') as f:
    f.write(summary_text)
print(summary_text)

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
fig.suptitle('GPU Resource Usage During Training (T4 x2)', fontsize=13)

styles = {0: dict(color='steelblue',   linestyle='-',  linewidth=1.5),
          1: dict(color='darkorange',   linestyle='--', linewidth=1.5)}

for gid in sorted(df_gpu['gpu'].unique()):
    sub = df_gpu[df_gpu['gpu'] == gid].sort_values('t')
    ax1.plot(sub['t'], sub['util_pct'], label=f'GPU {gid}', **styles.get(gid, {}))
    ax2.plot(sub['t'], sub['mem_mib'],  label=f'GPU {gid}', **styles.get(gid, {}))

ax1.set_ylabel('Utilization (%)')
ax1.set_title('GPU Utilization over Time')
ax1.set_ylim(0, 105)
ax1.yaxis.set_major_locator(ticker.MultipleLocator(20))
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.set_ylabel('Memory Used (MiB)')
ax2.set_title('GPU Memory over Time')
ax2.set_xlabel('Time (s)')
ax2.yaxis.set_major_locator(ticker.MultipleLocator(2000))
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/resource_usage.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: resource_usage.png  gpu_log.csv  gpu_log_parsed.tsv  gpu_stats.txt')
